<a href="https://colab.research.google.com/github/karsarobert/DeepLearning2026/blob/main/PTE_DL7_FELADATOK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Learning 2026 · 7. gyakorlat
## Colabban végzett, Inno által vezetett gyakorlat · 1.0

Témák: lakásár-regresszió, MNIST-számjegyfelismerés, autoencoderes tranzakciós anomáliadetektálás.

**Két felületet használunk:** a Colabban szerkesztesz és futtatsz, az Innoagentben beszélgetsz a tutorral.
Egy cellát dolgozz fel egyszerre. Minden cella előtt olvasd el a célt; futtatás után küldd vissza a kért eredményt.
Az agent nem látja automatikusan a Colabodat. Az „ok” helyett a tényleges kimenet rövid részletét másold át.

- **Kész cella:** a kód adott; olvasd el, futtasd, és értelmezd.
- **Önálló cella:** a szöveges feladat alapján te írod meg a kódot. Kérhetsz fokozatos segítséget.
- A hibás cellát javítsd, és csak utána haladj tovább. Kezdetben ne használd az „összes futtatása” műveletet.
- Az R_, M_ és A_ előtag a három feladat változóit különíti el.
- Az eredmények ábrái a cellák alatt jelennek meg. Nem kell az Innoagent fájlfáját vagy a teljes böngészőoldalt frissíteni.

Ez a kiinduló tanulói notebook átdolgozott munkapéldánya: külön validációval, javított cellasorrenddel és azonosítókkal.
A Boston-adatok történeti, korlátozott oktatási adatok; mai ingatlanértékelésre nem alkalmasak.


## Mintakódok használata

A korábbi első kódcella egymástól független részleteket tartalmazott. Itt ez nem egyben futtatandó program.
A részletes, más adatokon működő mintapéldákat a **DL_07_gyakorlat.html** fájlban találod.
Az alábbi rövid emlékeztetőkben az adatok már létrehozott, szemléltető változók.

```python
# Egy oszlop és egy kétdimenziós minta kiválasztásának általános alakja:
oszlop = tabla[: , 2]
egy_minta = tabla[4: 5]

# Meglévő skálázó használata új adaton:
uj_skala = skala.transform(uj_adatok)

# Más feladat egy sűrű rétege és egy dropout rétege:
reteg = layers.Dense(8, activation="relu")
eldobas = layers.Dropout(0.2)

# Kimeneti vektor legnagyobb elemének indexe:
index = np.argmax(egy_kimeneti_vektor)
```

A tömbök alakját, a cél formáját és a kimeneti neuronszámot mindig a saját feladathoz igazítsd.


## [E01] Környezet és munkamód

**Munkamód: kész cella.**

Ez a cella betölti a könyvtárakat és megadja a véletlenmagot. Nem tanít modellt. Az Innoagent beszélgetésre szolgál, a kód szerkesztése és futtatása a saját Colab-notebookodban történik. A külön R_, M_ és A_ változónevek megőrzik a három feladat eredményeit.

**Teendő:** Futtasd az importáló cellát a Colabban. Másold vissza a verziókat és a készenléti sort.

**Visszajelzés Innónak:** TensorFlow- és NumPy-verzió, valamint az E01 készenléti sor.


In [ ]:
# [E01]
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, classification_report
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from pathlib import Path

keras.utils.set_random_seed(42)
eredmenyek = {}
print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)
print("E01: a Colab-környezet készen áll.")


TensorFlow: 2.21.0
NumPy: 2.1.3
E01: a Colab-környezet készen áll.


## [R01] Lakásadatok betöltése

**Munkamód: kész cella.**

A történeti Boston-adatokban egy sor egy körzetet ír le, 13 jellemzővel. A cél a körzet lakásainak mediánértéke, ezer korabeli amerikai dollárban. A hivatalos NPZ-fájl betöltése után a következő lépésben saját, független felosztást készítünk. A helyi NPZ-fájl ellenőrzése csak letöltési tartalék: nem kell fájlt feltöltened, ha a Keras-letöltés működik.

**Teendő:** Futtasd a cellát. Írd meg a bemenet és a cél alakját, és mondd el, mi a célérték.

**Visszajelzés Innónak:** A két shape és a célérték saját szavas jelentése.


In [ ]:
# [R01]
R_adat_ut = Path("boston_housing.npz")
if not R_adat_ut.exists():
    R_adat_ut = Path(keras.utils.get_file(
        "boston_housing.npz",
        origin="https://storage.googleapis.com/tensorflow/tf-keras-datasets/boston_housing.npz",
        file_hash="f553886a1f8d56431e820c5b82552d9d95cfcb96d1e678153f8839538947dff5"
    ))
with np.load(R_adat_ut, allow_pickle=False) as adat:
    R_X = adat["x"].astype("float32")
    R_y = adat["y"].astype("float32")
print("R01 bemenet:", R_X.shape, "cél:", R_y.shape)
print("Első három célérték [ezer korabeli USD]:", R_y[:3])


57026/57026 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
R01 bemenet: (506, 13) cél: (506,)
Első három célérték [ezer korabeli USD]: [24.  21.6 34.7]


## [R02] Az adatok ellenőrzése

**Munkamód: kész cella.**

A DataFrame áttekinthető táblázatot készít. Az oszlopok minimuma és maximuma megmutatja az eltérő számtartományokat. Az isfinite azt ellenőrzi, hogy nincs-e NaN vagy végtelen érték.

**Teendő:** Futtasd a cellát. Nevezd meg két oszlop eltérő tartományát, és indokold, miért lesz hasznos a skálázás.

**Visszajelzés Innónak:** Két tényleges oszloptartomány és rövid indoklás.


In [ ]:
# [R02]
R_nevek = ["CRIM", "ZN", "INDUS", "CHAS", "NOX", "RM", "AGE",
           "DIS", "RAD", "TAX", "PTRATIO", "B", "LSTAT"]
R_tabla = pd.DataFrame(R_X, columns=R_nevek)
print(R_tabla.head(3))
print(pd.DataFrame({"minimum": R_tabla.min(), "maximum": R_tabla.max()}))
print("Minden érték véges:", np.isfinite(R_X).all(), np.isfinite(R_y).all())


      CRIM    ZN  INDUS  CHAS    NOX     RM        AGE     DIS  RAD    TAX  \
0  0.00632  18.0   2.31   0.0  0.538  6.575  65.199997  4.0900  1.0  296.0   
1  0.02731   0.0   7.07   0.0  0.469  6.421  78.900002  4.9671  2.0  242.0   
2  0.02729   0.0   7.07   0.0  0.469  7.185  61.099998  4.9671  2.0  242.0   

     PTRATIO           B  LSTAT  
0  15.300000  396.899994   4.98  
1  17.799999  396.899994   9.14  
2  17.799999  392.829987   4.03  
           minimum     maximum
CRIM       0.00632   88.976196
ZN         0.00000  100.000000
INDUS      0.46000   27.740000
CHAS       0.00000    1.000000
NOX        0.38500    0.871000
RM         3.56100    8.780000
AGE        2.90000  100.000000
DIS        1.12960   12.126500
RAD        1.00000   24.000000
TAX      187.00000  711.000000
PTRATIO   12.60000   22.000000
B          0.32000  396.899994
LSTAT      1.73000   37.970001
Minden érték véges: True True


## [R03] Tanító-, validációs és teszthalmaz

**Munkamód: kész cella.**

Először 60% tanítóadatot választunk, majd a maradékot felezzük. A validáció a korai leállításban segít, a végső tesztet a tanítás befejezése után értékeljük. A kerekítés miatt nem teljesen azonos a validációs és tesztdarabszám.

**Teendő:** Futtasd a cellát. Küldd el a három darabszámot, és nevezd meg, melyik halmaz alapján állítjuk le a tanítást.

**Visszajelzés Innónak:** Három darabszám és a validáció szerepe.


In [ ]:
# [R03]
R_X_train, R_X_maradek, R_y_train, R_y_maradek = train_test_split(
    R_X, R_y, test_size=0.4, random_state=42
)
R_X_val, R_X_test, R_y_val, R_y_test = train_test_split(
    R_X_maradek, R_y_maradek, test_size=0.5, random_state=42
)
print("R03 tanító / validáció / teszt:",
      len(R_X_train), len(R_X_val), len(R_X_test))


R03 tanító / validáció / teszt: 303 101 102


## [R04] Skálázás adatszivárgás nélkül

**Munkamód: önálló cella.**

A skálázó oszloponként tanul minimumot és maximumot. Ezt csak a tanítóbemenetből szabad meghatározni. A célérték most eredeti egységben marad.

**Teendő:** Hozz létre R_scaler néven MinMaxScaler objektumot. Készítsd el R_X_train_s, R_X_val_s és R_X_test_s tömböket. Csak a tanítón használj fit_transform műveletet, a többi halmazon transformot. Írasd ki mindhárom shape-et és a skálázó n_samples_seen_ értékét.

**Visszajelzés Innónak:** A saját skálázó kódod, a három shape és az illesztési mintaszám.


In [ ]:
# [R04]
# Önálló kódírás: Skálázás adatszivárgás nélkül
# Írd meg a fenti leírás szerinti kódot.
# Használd a leírásban megadott változóneveket.
R_scaler = MinMaxScaler()

R_X_train_s = R_scaler.fit_transform(R_X_train)
R_X_val_s = R_scaler.transform(R_X_val)
R_X_test_s = R_scaler.transform(R_X_test)

print("R04 shape-ek:",
      R_X_train_s.shape, R_X_val_s.shape, R_X_test_s.shape)
print("Illesztett minták:", R_scaler.n_samples_seen_)



R04 shape-ek: (303, 13) (101, 13) (102, 13)
Illesztett minták: 303


## [R05] Regressziós modell

**Munkamód: önálló cella.**

A hálózat 13 jellemzőből egy valós értéket becsül. A rejtett rétegek ReLU aktivációt használnak. A lineáris kimenet nem korlátozza 0 és 1 közé az árbecslést.

**Teendő:** Építs R_model nevű Sequential modellt: a bemenetet a skálázott tanítóadat jellemzőszáma határozza meg; utána 64, majd 32 ReLU-neuron következzen, végül egy lineáris kimenet. Jelenítsd meg a summary-t.

**Visszajelzés Innónak:** Modellkód és a summary utolsó rétegének alakja; miért egy kimenet?


In [ ]:
# [R05]
# Önálló kódírás: Regressziós modell
# Írd meg a fenti leírás szerinti kódot.
# Használd a leírásban megadott változóneveket.


## [R06] Veszteség és korai leállítás

**Munkamód: önálló cella.**

A compile nem tanít, hanem beállítja az optimalizálót és a veszteséget. A MAE az abszolút eltérések átlaga. A korai leállítás a validációs veszteség alapján a tanulás megfelelő állapotát választja.

**Teendő:** Fordítsd R_model-t adam optimalizálóval és mae veszteséggel. Hozz létre R_stop EarlyStopping objektumot: val_loss figyelése, min_delta=0.001, patience=12, legjobb súlyok visszaállítása. Írd ki a loss-t és a monitor nevét.

**Visszajelzés Innónak:** Compile- és callback-kód; mi történik 12 javulás nélküli epoch után?


In [ ]:
# [R06]
# Önálló kódírás: Veszteség és korai leállítás
# Írd meg a fenti leírás szerinti kódot.
# Használd a leírásban megadott változóneveket.


## [R07] Regresszió tanítása

**Munkamód: kész cella.**

A fit a tanítómintákból és áraikból módosítja a súlyokat. A validáció nem kap közvetlen súlyfrissítést. A 150 epoch felső korlát; a callback ennél korábban is leállíthat.

**Teendő:** Futtasd a tanítást, várd meg a végét. Küldd el a lefutott epochok számát és a legkisebb validációs veszteséget.

**Visszajelzés Innónak:** Az R07 összefoglaló két száma a tanítás befejeződése után.


In [ ]:
# [R07]
R_history = R_model.fit(
    R_X_train_s, R_y_train,
    validation_data=(R_X_val_s, R_y_val),
    epochs=150, batch_size=32, callbacks=[R_stop], verbose=1
)
print("R07 lefutott epochok:", len(R_history.history["loss"]))
print("Legkisebb validációs MAE:", min(R_history.history["val_loss"]))


## [R08] Tanulási görbék értelmezése

**Munkamód: kész cella.**

A grafikon a tanító és validációs MAE-t mutatja. A görbék közös csökkenése és tartós szétválása más tanulási helyzetre utal. Pontos epochszám helyett az irány és a kapcsolat felismerése a fontos.

**Teendő:** Futtasd a rajzolást. Írj le egy konkrét megfigyelést a tanító- és validációs görbe kapcsolatáról.

**Visszajelzés Innónak:** Képernyőkép vagy saját szavas, konkrét görbeleírás.


In [ ]:
# [R08]
plt.figure(figsize=(8, 4))
plt.plot(R_history.history["loss"], label="Tanító MAE")
plt.plot(R_history.history["val_loss"], label="Validációs MAE")
plt.xlabel("Epoch indexe (0-tól)")
plt.ylabel("MAE [ezer korabeli USD]")
plt.legend()
plt.tight_layout()
plt.show()


## [R09] Végső teszt és alapbecslés

**Munkamód: kész cella.**

A tesztet most használjuk először eredményértékelésre. Összevetjük a hálózatot egy állandó alapbecsléssel, amely mindig a tanítócélok mediánját tippeli. MAE-nél ez természetes kiindulás.

**Teendő:** Futtasd a cellát. Küldd el a két MAE-t, és mondd el a hálózat hibájának jelentését a megfelelő egységben.

**Visszajelzés Innónak:** Hálózati MAE, alap-MAE, és a hálózati szám jelentése.


In [ ]:
# [R09]
R_mae = float(R_model.evaluate(R_X_test_s, R_y_test, verbose=0))
R_pred = R_model.predict(R_X_test_s, verbose=0).ravel()
R_alap = np.full_like(R_y_test, np.median(R_y_train))
R_alap_mae = mean_absolute_error(R_y_test, R_alap)
print(f"R09 hálózat MAE: {R_mae:.3f} ezer korabeli USD")
print(f"Alapbecslés MAE: {R_alap_mae:.3f} ezer korabeli USD")
print(pd.DataFrame({"valos": R_y_test[:5], "becsult": R_pred[:5]}))
eredmenyek["regresszio"] = {"mae": R_mae, "alap_mae": float(R_alap_mae)}


## [R10] Új adat és ugyanaz a skálázó

**Munkamód: önálló cella.**

A teszthalmaz egy nyers mintáján szemléltetjük az új adat útját. Ez nem új független teszt; az egyedi becslés lépéseit nézzük meg. A mintaszám dimenzióját egyetlen sornál is megtartjuk.

**Teendő:** Válaszd az R_X_test első sorát kétdimenziós szelettel R_uj néven. A már illesztett R_scaler segítségével készítsd el R_uj_s-t, majd számíts becslést R_model-lel. Írd ki a bemenet és a kimenet alakját és a becslést.

**Visszajelzés Innónak:** Kód, a két alak és annak indoklása, miért nem fit_transform szerepel.


In [ ]:
# [R10]
# Önálló kódírás: Új adat és ugyanaz a skálázó
# Írd meg a fenti leírás szerinti kódot.
# Használd a leírásban megadott változóneveket.


## [M01] MNIST és egy gyorsabb tanító részhalmaz

**Munkamód: kész cella.**

Az MNIST kézzel írt számjegyek 28×28-as képeit tartalmazza. A gyorsabb órai munka érdekében a 60000 tanítómintából 12000-et választunk osztályarányt tartó véletlen részhalmazként: 10000 tanító és 2000 validációs képet. Az eredeti 10000 tesztképet elkülönítve hagyjuk.

**Teendő:** Futtasd a betöltést. Küldd el a három képhalmaz alakját és a címkék értéktartományát.

**Visszajelzés Innónak:** Három shape és a címkék tartománya.


In [ ]:
# [M01]
if Path("mnist.npz").exists():
    with np.load("mnist.npz", allow_pickle=False) as adat:
        M_teljes_kepek = adat["x_train"]
        M_teljes_cimkek = adat["y_train"]
        M_test_kepek = adat["x_test"]
        M_y_test = adat["y_test"]
else:
    (M_teljes_kepek, M_teljes_cimkek), (M_test_kepek, M_y_test) = (
        keras.datasets.mnist.load_data()
    )
M_resz_kepek, _, M_resz_cimkek, _ = train_test_split(
    M_teljes_kepek, M_teljes_cimkek, train_size=12000,
    random_state=42, stratify=M_teljes_cimkek
)
M_train_kepek, M_val_kepek, M_y_train, M_y_val = train_test_split(
    M_resz_kepek, M_resz_cimkek, test_size=2000,
    random_state=42, stratify=M_resz_cimkek
)
print("M01 tanító:", M_train_kepek.shape)
print("Validáció:", M_val_kepek.shape, "teszt:", M_test_kepek.shape)
print("Címkék:", np.unique(M_y_train))


## [M02] Egy kép és a valódi címkéje

**Munkamód: kész cella.**

A 3-as index a negyedik képet jelenti. M_y_test még egész osztályindexeket tárol. A címke azt jelzi, milyen számjegyet ábrázol a kép az adatkészlet szerint.

**Teendő:** Futtasd a cellát. Mondd meg, melyik sorszámú képet nézzük, és mi a valódi címkéje.

**Visszajelzés Innónak:** A kép sorszáma és a saját kimenetében látható egész címke.


In [ ]:
# [M02]
M_index = 3  # A negyedik tesztkép, nulláról indexelve.
plt.imshow(M_test_kepek[M_index], cmap="gray", vmin=0, vmax=255)
plt.title(f"Valódi címke: {M_y_test[M_index]}")
plt.axis("off")
plt.show()
print("M02 index:", M_index, "címke:", int(M_y_test[M_index]))


## [M03] Normalizálás és kilapítás

**Munkamód: önálló cella.**

A képpontok uint 8 értékei 0 és 255 közöttiek. Float 32-re alakítva és 255-tel osztva 0–1 tartományt kapunk. A 28×28 kép 784 eleme a reshape során ugyanaz marad, csak vektorba rendezzük.

**Teendő:** A nyers M_train_kepek, M_val_kepek és M_test_kepek tömbökből készíts M_X_train, M_X_val és M_X_test float 32 tömböket. Oszd az értékeket 255-tel, és alakítsd mindegyik képet 784 elemű vektorrá. Írd ki a három shape-et, valamint a tanítóbemenet minimumát és maximumát.

**Visszajelzés Innónak:** A kód, a három shape, a tartomány és a -1 jelentése.


In [ ]:
# [M03]
# Önálló kódírás: Normalizálás és kilapítás
# Írd meg a fenti leírás szerinti kódot.
# Használd a leírásban megadott változóneveket.


## [M04] One-hot célvektorok

**Munkamód: önálló cella.**

A one-hot vektor hossza az osztályok száma. A helyes osztály helyén 1, a többi helyen 0 van. Az eredeti M_y_train, M_y_val és M_y_test tömböket nem írjuk felül.

**Teendő:** A keras.utils.to_categorical segítségével készíts M_y_train_oh, M_y_val_oh és M_y_test_oh tömböket tíz osztállyal. Írd ki a tanítócél alakját és az első egész címkét a hozzá tartozó one-hot vektorral.

**Visszajelzés Innónak:** A kód, (10000, 10) alak és egy összetartozó címkepár.


In [ ]:
# [M04]
# Önálló kódírás: One-hot célvektorok
# Írd meg a fenti leírás szerinti kódot.
# Használd a leírásban megadott változóneveket.


## [M05] Számjegyosztályozó hálózat

**Munkamód: önálló cella.**

A 784 bemeneti számhoz két sűrű rejtett réteg kapcsolódik. A dropout tanításkor egyes aktivációkat véletlenszerűen nulláz. A tíz softmax-kimenet a tíz számjegyhez tartozik.

**Teendő:** Építs M_model néven hálózatot 784 bemenettel, 128 ReLU-neuronnal, 0.3 dropouttal, 64 ReLU-neuronnal, újabb 0.3 dropouttal és tíz softmax-kimenettel. Jelenítsd meg a summary-t.

**Visszajelzés Innónak:** Modellkód és a summary utolsó alakja; a tíz kimenet indoka.


In [ ]:
# [M05]
# Önálló kódírás: Számjegyosztályozó hálózat
# Írd meg a fenti leírás szerinti kódot.
# Használd a leírásban megadott változóneveket.


## [M06] Osztályozási veszteség

**Munkamód: önálló cella.**

A címkék one-hot alakúak, a modell softmax-kimenetet ad. Ehhez categorical_crossentropy illik. Az accuracy a helyes osztályválasztások aránya, és nem azonos a veszteséggel.

**Teendő:** Fordítsd M_model-t adam optimalizálóval, a one-hot címkékhez megfelelő veszteséggel és accuracy metrikával. Hozd létre M_stop callbacket val_loss figyeléssel, patience=4 és a legjobb súlyok visszaállításával. Írd ki a veszteség nevét.

**Visszajelzés Innónak:** Compile- és callback-kód; a választott veszteség indoklása.


In [ ]:
# [M06]
# Önálló kódírás: Osztályozási veszteség
# Írd meg a fenti leírás szerinti kódot.
# Használd a leírásban megadott változóneveket.


## [M07] Az osztályozó tanítása

**Munkamód: kész cella.**

Legfeljebb húsz epochot engedünk, 256-os batchmérettel. A validációs képek és célvektorok külön halmazban vannak. A verbose=1 mutatja az előrehaladást; a végső teszt még nem vesz részt.

**Teendő:** Futtasd a tanítást, várd meg a végét. Küldd el az epochok számát és a legnagyobb validációs accuracy-t.

**Visszajelzés Innónak:** Epochszám és a validációs accuracy saját kimenetből.


In [ ]:
# [M07]
M_history = M_model.fit(
    M_X_train, M_y_train_oh,
    validation_data=(M_X_val, M_y_val_oh),
    epochs=20, batch_size=256, callbacks=[M_stop], verbose=1
)
print("M07 epochok:", len(M_history.history["loss"]))
print("Legnagyobb validációs accuracy:", max(M_history.history["val_accuracy"]))


## [M08] Veszteség és accuracy külön ábrán

**Munkamód: kész cella.**

Balra a keresztentrópia, jobbra a találati arány látható. A dropout tanításkor aktív, validációnál nem, ezért a validációs görbe helyzete önmagában nem mutat hibát.

**Teendő:** Futtasd az ábrázolást, és a két ábra közül egyen írj le konkrét tanító–validációs megfigyelést.

**Visszajelzés Innónak:** Kép vagy konkrét, a saját ábrára vonatkozó leírás.


In [ ]:
# [M08]
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(M_history.history["loss"], label="Tanító")
ax[0].plot(M_history.history["val_loss"], label="Validáció")
ax[0].set_title("Keresztentrópia")
ax[1].plot(M_history.history["accuracy"], label="Tanító")
ax[1].plot(M_history.history["val_accuracy"], label="Validáció")
ax[1].set_title("Accuracy")
for tengely in ax:
    tengely.set_xlabel("Epoch indexe (0-tól)")
    tengely.legend()
plt.tight_layout()
plt.show()


## [M09] Teszteredmény és egyetlen számjegy jóslata

**Munkamód: önálló cella.**

A predict egy képre tíz kimeneti számot ad. Ezek maximumának helye a becsült számjegy. A valódi címke az eredeti egész M_y_test tömbből olvasható ki.

**Teendő:** Értékeld M_model-t M_X_test és M_y_test_oh alapján; tárold az eredményt M_loss és M_acc néven. A korábban választott M_index képét kétdimenziós szelettel add a predictnek, majd np.argmax segítségével válaszd ki a becsült számjegyet M_josolt néven. Írd ki az accuracy-t, a tíz kimenetet, a becsült és a valódi címkét.

**Visszajelzés Innónak:** Kód; accuracy; kimeneti vektor; becsült és valódi számjegy.


In [ ]:
# [M09]
# Önálló kódírás: Teszteredmény és egyetlen számjegy jóslata
# Írd meg a fenti leírás szerinti kódot.
# Használd a leírásban megadott változóneveket.


## [M10] Tévesztések vizsgálata

**Munkamód: kész cella.**

A teljes teszt kimenete 10000×10. Az argmax(axis=1) soronként, azaz képenként választ osztályt. A mátrix sorai valós, oszlopai becsült címkék. A kód egy hibás példát mutat, ha van ilyen.

**Teendő:** Futtasd a cellát. Értelmezz egy tévesztési mátrixcellát valódi címkével, becsült címkével és darabszámmal.

**Visszajelzés Innónak:** Egy tényleges mátrixcella jelentése vagy egy hibás kép címkepárja; teljes teszt hibaszáma.


In [ ]:
# [M10]
M_prob_mind = M_model.predict(M_X_test, verbose=0)
M_pred = np.argmax(M_prob_mind, axis=1)
M_cm = confusion_matrix(M_y_test, M_pred, labels=np.arange(10))
ConfusionMatrixDisplay(M_cm, display_labels=np.arange(10)).plot(cmap="Blues")
plt.title("Valós címke: sor; becsült címke: oszlop")
plt.xlabel("Becsült címke")
plt.ylabel("Valós címke")
plt.tight_layout()
plt.show()
M_hibas = np.flatnonzero(M_pred != M_y_test)
print("M10 tévesztések száma:", len(M_hibas))
if len(M_hibas):
    i = int(M_hibas[0])
    plt.imshow(M_test_kepek[i], cmap="gray")
    plt.title(f"Valós: {M_y_test[i]}, becsült: {M_pred[i]}")
    plt.axis("off")
    plt.show()
eredmenyek["mnist"] = {"accuracy": float(M_acc), "hibak": int(len(M_hibas))}


## [A01] Szintetikus tranzakciók és a címkék

**Munkamód: kész cella.**

A kiinduló notebookhoz hasonlóan 1000 normál és 50 eltolással képzett rendellenes, öt jellemzőjű mintát generálunk. A jellemzők itt mesterséges számok, nem valós tranzakciós mezők. A címke 0 normál, az 1 rendellenes. Ez az egyszerű adatkészlet nem reprezentálja a valós banki csalások összetettségét.

**Teendő:** Futtasd a cellát. Írd le a két osztály darabszámát, és nevezd meg, melyik címke jelenti az anomáliát.

**Visszajelzés Innónak:** Címkejelentés, darabszámok és bemeneti shape.


In [ ]:
# [A01]
A_rng = np.random.default_rng(42)
A_normal = A_rng.random((1000, 5)) * 10
A_anomalia = A_rng.random((50, 5)) * 10 + 5
A_X = np.vstack([A_normal, A_anomalia]).astype("float32")
A_y = np.concatenate([np.zeros(1000), np.ones(50)]).astype("int32")
A_ertekek, A_darabok = np.unique(A_y, return_counts=True)
print("A01 címkék és darabok:", dict(zip(A_ertekek, A_darabok)))
print("Bemenet:", A_X.shape)


## [A02] Felosztás a skálázás előtt

**Munkamód: kész cella.**

A stratify megőrzi az osztályarányokat. Még nyers adatokkal dolgozunk: nincs a teljes adatra illesztett skálázó. A normál mintákat csak a felosztás után választjuk ki.

**Teendő:** Futtasd a felosztást. Küldd el a három elemszámot és a teszt két osztályának darabszámát.

**Visszajelzés Innónak:** 630/210/210 és a teszt címkeeloszlása.


In [ ]:
# [A02]
A_X_train, A_X_maradek, A_y_train, A_y_maradek = train_test_split(
    A_X, A_y, test_size=0.4, random_state=42, stratify=A_y
)
A_X_val, A_X_test, A_y_val, A_y_test = train_test_split(
    A_X_maradek, A_y_maradek, test_size=0.5,
    random_state=42, stratify=A_y_maradek
)
print("A02 darabszámok:", len(A_X_train), len(A_X_val), len(A_X_test))
print("Teszt címkeeloszlás:", np.unique(A_y_test, return_counts=True))


## [A03] Csak normál tanítóadat és külön kalibráció

**Munkamód: önálló cella.**

A 0 címke alapján választjuk ki a normál sorokat. A normál validációs adatot két részre bontjuk: az egyik a korai leállításhoz, a másik a küszöb kalibrálásához szolgál. Így a küszöb még a validáció tanulást irányító részétől is elkülönül.

**Teendő:** Készíts A_normal_train és A_normal_val tömböket a 0 címkéjű tanító-, illetve validációs sorokból. A_normal_val-t train_test_split segítségével felezd A_normal_stop és A_normal_cal tömbre, random_state=42 mellett. Írd ki a három felhasznált rész alakját.

**Visszajelzés Innónak:** Saját szűrőkód és a három shape.


In [ ]:
# [A03]
# Önálló kódírás: Csak normál tanítóadat és külön kalibráció
# Írd meg a fenti leírás szerinti kódot.
# Használd a leírásban megadott változóneveket.


## [A04] A normál működés skálázása

**Munkamód: önálló cella.**

Ugyanazt a skálázót alkalmazzuk a normál stop-, normál kalibrációs és vegyes tesztadatokra. Az anomáliák skálázott értékei túlléphetnek az 1-en: ezt nem vágjuk le, mert az eltérés információt hordozhat.

**Teendő:** Hozz létre A_scaler MinMaxScaler objektumot, és készítsd el az A_train_s, A_stop_s, A_cal_s, A_test_s tömböket. Az illesztés csak A_normal_train-en történjen. Írd ki a skálázó illesztési mintaszámát és a négy shape-et.

**Visszajelzés Innónak:** Kód, 600-as illesztési mintaszám és négy shape.


In [ ]:
# [A04]
# Önálló kódírás: A normál működés skálázása
# Írd meg a fenti leírás szerinti kódot.
# Használd a leírásban megadott változóneveket.


## [A05] Autoencoder szűk keresztmetszettel

**Munkamód: önálló cella.**

Öt jellemzőt próbálunk visszaállítani. A középső három érték kisebb dimenziójú, mint a bemenet. Az eredeti mesterséges normál adatok nem eleve alacsony dimenziósak, ezért a tömörítés a normál adatoknál is okozhat hibát. Az öt sigmoid kimenet a normál tanítóadat 0–1 tartományához illik; a tartományon kívüli bemenetet ezért nem tudja hibátlanul visszaadni.

**Teendő:** Építs A_model nevű autoencodert: 5 bemenet → 16 ReLU → 3 lineáris → 16 ReLU → 5 sigmoid kimenet. Fordítsd adam optimalizálóval és mse veszteséggel. Jelenítsd meg a summary-t.

**Visszajelzés Innónak:** Modellkód; az öt kimenet és a háromelemű középső réteg jelentése.


In [ ]:
# [A05]
# Önálló kódírás: Autoencoder szűk keresztmetszettel
# Írd meg a fenti leírás szerinti kódot.
# Használd a leírásban megadott változóneveket.


## [A06] A bemenet tanítása önmagára

**Munkamód: kész cella.**

A tanító bemenet és cél ugyanaz a normál tömb. A stop-halmaz is csak normál adatot tartalmaz. A kalibrációs adat még nem szerepel a fit-ben, a teszt sem.

**Teendő:** Futtasd a tanítást. Küldd el az epochszámot, és mondd el, miért szerepel ugyanaz a tömb a fit első két argumentumában.

**Visszajelzés Innónak:** Epochszám és a bemenet=cél magyarázata.


In [ ]:
# [A06]
A_stop = callbacks.EarlyStopping(
    monitor="val_loss", patience=10, restore_best_weights=True
)
A_history = A_model.fit(
    A_train_s, A_train_s,
    validation_data=(A_stop_s, A_stop_s),
    epochs=100, batch_size=32, callbacks=[A_stop], verbose=1
)
print("A06 epochok:", len(A_history.history["loss"]))
print("Legkisebb normál validációs MSE:", min(A_history.history["val_loss"]))


## [A07] Mintánkénti kalibrációs hiba

**Munkamód: önálló cella.**

Először a külön normál kalibrációs mintákat rekonstruáljuk. Az öt jellemző négyzetes eltérését soronként átlagoljuk, így egy mintához egy hibaszám tartozik. A teljes tömb egyetlen átlaga nem alkalmas mintánkénti döntésre.

**Teendő:** Számítsd ki A_cal_rek-et A_model.predict-tel az A_cal_s-ből. Ezután készíts A_cal_hiba néven mintánkénti MSE-t np.mean segítségével, a megfelelő tengely mentén. Írd ki a rekonstrukció és a hibatömb alakját, valamint az első öt hibát.

**Visszajelzés Innónak:** Kód és(100, 5)rekonstrukció, (100, )hiba, axisindoklás.


In [ ]:
# [A07]
# Önálló kódírás: Mintánkénti kalibrációs hiba
# Írd meg a fenti leírás szerinti kódot.
# Használd a leírásban megadott változóneveket.


## [A08] Küszöb meghatározása a teszt előtt

**Munkamód: önálló cella.**

A normál kalibrációs hibák 97%-os kvantilisét választjuk. Ez előre rögzített oktatási szabály, nem biztosít minden új halmazon pontosan 3% téves riasztást. Az átlag+2 szórás másik lehetséges heurisztika, nem kötelező matematikai törvény.

**Teendő:** Készíts A_kuszob változót az A_cal_hiba 97%-os kvantiliseként np.quantile segítségével. Írd ki a küszöböt és a kalibrációs hibák számát; mondd el, miért nem a teszthibákat használjuk.

**Visszajelzés Innónak:** Kód, küszöb, 100 kalibrációs minta; a teszt függetlenségének indoka.


In [ ]:
# [A08]
# Önálló kódírás: Küszöb meghatározása a teszt előtt
# Írd meg a fenti leírás szerinti kódot.
# Használd a leírásban megadott változóneveket.


## [A09] Vegyes teszt: döntések és metrikák

**Munkamód: önálló cella.**

A210 tesztminta között normál és rendellenes adat is van. A rögzített küszöbnél nagyobb hiba 1-es döntést ad. A címkéket csak az eredményértékelésben hasonlítjuk a döntésekhez.

**Teendő:** Rekonstruáld A_test_s-t A_test_rek néven, számíts A_test_hiba mintánkénti MSE-t, majd készíts A_pred 0/1 egész döntéseket a rögzített A_kuszob segítségével. Írasd ki classification_report-tal a Normál és Anomália osztályok eredményét, labels=[0, 1], zero_division=0 mellett.

**Visszajelzés Innónak:** Kód, classification_report, az Anomália sor precisionje és recallja.


In [ ]:
# [A09]
# Önálló kódírás: Vegyes teszt: döntések és metrikák
# Írd meg a fenti leírás szerinti kódot.
# Használd a leírásban megadott változóneveket.


## [A10] Hibaeloszlás és tévesztések

**Munkamód: kész cella.**

A mátrix összefoglalja a helyes és hibás döntéseket. A hibaeloszlás a két valódi osztály rekonstrukciós hibáit mutatja, a függőleges vonal pedig a korábban rögzített küszöb. Itt már nem állítunk a küszöbön.

**Teendő:** Futtasd a cellát. Küldd vissza a TN, FP, FN, TP értékeket, és magyarázd el egy téves riasztás vagy elmulasztott anomália jelentését.

**Visszajelzés Innónak:** Négy darabszám és konkrét értelmezés; a küszöb nem változott.


In [ ]:
# [A10]
A_cm = confusion_matrix(A_y_test, A_pred, labels=[0, 1])
A_tn, A_fp, A_fn, A_tp = A_cm.ravel()
print("A10 TN, FP, FN, TP:", A_tn, A_fp, A_fn, A_tp)
ConfusionMatrixDisplay(A_cm, display_labels=["Normál", "Anomália"]).plot(cmap="Blues")
plt.title("Valós címke: sor; becsült címke: oszlop")
plt.xlabel("Becsült címke")
plt.ylabel("Valós címke")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
A_bins = np.histogram_bin_edges(A_test_hiba, bins=25)
plt.hist(A_test_hiba[A_y_test == 0], bins=A_bins, alpha=0.65, label="Normál")
plt.hist(A_test_hiba[A_y_test == 1], bins=A_bins, alpha=0.65, label="Anomália")
plt.axvline(A_kuszob, color="black", linestyle="--", label="Rögzített küszöb")
plt.xlabel("Mintánkénti rekonstrukciós MSE")
plt.ylabel("Minták száma")
plt.legend()
plt.tight_layout()
plt.show()

A_precision = float(A_tp / (A_tp + A_fp)) if A_tp + A_fp else 0.0
A_recall = float(A_tp / (A_tp + A_fn)) if A_tp + A_fn else 0.0
eredmenyek["anomalia"] = {"precision": A_precision, "recall": A_recall,
                        "TN": int(A_tn), "FP": int(A_fp),
                        "FN": int(A_fn), "TP": int(A_tp)}


## [Z01] Eredmények mentése és lezárás

**Munkamód: kész cella.**

A három feladat eltérő célértéket és metrikát használ. A rövid összefoglaló segít a saját futási eredményeket egy helyen áttekinteni. A kitöltött notebookot a Colab mentési vagy letöltési lehetőségével őrizd meg.

**Teendő:** Futtasd a cellát, küldd el az összefoglalót, és mentsd a saját kitöltött notebookodat.

**Visszajelzés Innónak:** A három feladat saját eredményeinek összesítése és a mentés jelzése.


In [ ]:
# [Z01]
print("Z01: saját futások eredményei")
for feladat, adatok in eredmenyek.items():
    print(feladat, adatok)
print("Mentsd a kitöltött notebookot a kódokkal és kimenetekkel együtt.")


## Rövid záróbeszélgetés

Inno egyenként kérdez majd:

1. Miben különbözött a regresszió, az osztályozás és az autoencoder célértéke?
2. Miért nem a végső teszthalmazból választottunk skálázást, modellt vagy küszöböt?
3. Mit jelent a saját anomáliaeredményed precisionje és recallja?

Nincs további kötelező feladat vagy házi feladat. Ha egy rész kimaradt, azt a tutorral együtt pontosan jelöljétek.
